# Nick 4.8 — QLoRA training (Approx AI)

Run on **Kaggle** (Settings → Accelerator → GPU T4/P100, Internet ON) or **Google Colab** (GPU runtime).

Base: `Qwen/Qwen2.5-3B-Instruct` · Method: QLoRA (4-bit NF4 + LoRA r=16) · Output: PEFT adapter + SHA-256 manifest + verification report.

**Training is complete only when `verify_adapter.py` prints `verification_passed: true`.**

In [ ]:
!nvidia-smi
!git clone https://github.com/umer630/Approx-Ai.git approx || (cd approx && git pull)
%cd approx/training
!pip install -q -r requirements.txt

## 1. Data
Optional: attach approved Kaggle datasets (after license review) and convert them to `data/external/*.jsonl` in the Approx message schema, then set `approved: true` in `configs/dataset_sources.json`.
If you exported consented user history from the Approx admin API, place it at `data/consented/latest.jsonl`.

In [ ]:
!python data/seed/make_seed.py
!python -m approx_data.build --version nick-sft-0.1.0
!cat data/processed/manifest.json | head -60

## 2. Train (auto-resumes from the last checkpoint)
On T4/P100 (no bf16) the script automatically switches to fp16 compute.

In [ ]:
!python train_qlora.py --config configs/nick-4.8-qlora.json

## 3. Verify (files, hashes, adapter load, inference, identity/language/safety probes, perplexity)

In [ ]:
!python verify_adapter.py --adapter outputs/nick-4.8-qlora-v1/adapter
!cat outputs/nick-4.8-qlora-v1/adapter/verification.json | head -80

## 4. Package
The adapter is a separate artifact (not committed to git). Save it as a Kaggle Dataset / Model or download the tarball.

In [ ]:
!cd outputs/nick-4.8-qlora-v1 && tar czf nick-4.8-adapter-1.0.0.tar.gz adapter && sha256sum nick-4.8-adapter-1.0.0.tar.gz | tee nick-4.8-adapter-1.0.0.tar.gz.sha256

## 5. Serve
```
vllm serve Qwen/Qwen2.5-3B-Instruct --enable-lora --lora-modules nick-4.8=outputs/nick-4.8-qlora-v1/adapter --port 8000
```
Then configure the Approx backend: `APPROX_MODEL_NICK_URL=http://<host>:8000/v1`, `APPROX_MODEL_NICK_NAME=nick-4.8`.
Register the adapter: `POST /api/admin/training/adapters` with the manifest SHA-256 and `verification.json`.